# Day 12 — D2 Receptor + Levetiracetam Production MD

**Goal:** Run 50 ns of MD on the D2 + levetiracetam docked complex in DPPC membrane.
Test whether the **ASP80 H-bond** (predicted from Day 11 docking) holds dynamically.

**Strategy (why this version will work where earlier attempts failed):**

Earlier attempts rebuilt the force field via `SystemGenerator(forcefields=['charmm36.xml', ...])`.
That fails because OpenMM's `charmm36.xml` does **not** include DPPC lipid residues, CMAP
corrections, or NBFix — those parameters live inside the `gmx_top.pkl` (ParmEd Structure)
we created on Day 8 from MemProtMD's GROMACS files.

This notebook **merges at the ParmEd level** instead:

```
   gmx_top.pkl (CHARMM36: D2 + DPPC + water + ions)  ──┐
                                                        ├─► combined ParmEd Structure
   levetiracetam (OpenFF-2.1.0 parameters)         ────┘                │
                                                                        ▼
                                                            combined.createSystem()
                                                                        │
                                                                        ▼
                                                       OpenMM System (all 9 forces preserved)
```

**Inputs (must exist on `/workspace/d2_system/`):**
| File | What it is |
|---|---|
| `gmx_top.pkl` | ParmEd Structure of D2 + DPPC membrane (CHARMM36) |
| `d2_membrane_topology.pdb` | Topology + initial positions |
| `day9_final_equilibrated.xml` | Equilibrated OpenMM State (positions + box + velocities) |
| `levetiracetam_docked.sdf` | **NEW — the Day 11 docked pose, exported as SDF with 3D coords** |

If you have the docked pose only as PDBQT, Cell 3b converts it to SDF.

**Timing on RTX PRO 4500 Blackwell (118k-atom system):**
- Setup + minimize + 2 ns equilibration: ~20–30 min
- 50 ns production: ~10–15 hours (run overnight)


---
## Part 1 — Environment check

Run this first. If any line fails, fix it before going further.
The notebook assumes the `mddock` conda env is active.


In [ ]:
# Cell 1: Imports and version sanity check
import sys, os, time, pickle
from pathlib import Path

import numpy as np

import openmm
from openmm import (
    XmlSerializer, LangevinMiddleIntegrator, MonteCarloMembraneBarostat,
    Platform, CustomExternalForce, Vec3,
)
from openmm import unit as u
from openmm import app

import parmed as pmd
from openff.toolkit import Molecule, ForceField as OFFForceField

print("Python      :", sys.version.split()[0])
print("OpenMM      :", openmm.__version__)
print("ParmEd      :", pmd.__version__)
import openff.toolkit
print("OpenFF tk   :", openff.toolkit.__version__)
import openff.interchange
print("Interchange :", openff.interchange.__version__)

# GPU check
platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print("Platforms   :", platforms)
assert "CUDA" in platforms, "CUDA platform not found — check GPU + OpenMM CUDA install"
print("✓ CUDA available")


In [ ]:
# Cell 2: Working directory and input file check
WORKDIR = Path("/workspace/d2_system")
os.chdir(WORKDIR)
print("CWD:", Path.cwd())

REQUIRED = {
    "gmx_top.pkl"                  : "ParmEd Structure of D2 + DPPC membrane",
    "d2_membrane_topology.pdb"     : "Topology + initial positions",
    "day9_final_equilibrated.xml"  : "Equilibrated OpenMM State",
     "docking/best_pose_equilibrated_ligand.pdb" : "Day 11 docked pose (PDB)",
}

missing = []
for name, desc in REQUIRED.items():
    p = WORKDIR / name
    if p.exists():
        print(f"  ✓ {name:35s}  ({p.stat().st_size/1e6:.1f} MB)   — {desc}")
    else:
        print(f"  ✗ {name:35s}  MISSING — {desc}")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing inputs: {missing}\n"
        "Run Cell 3b (PDBQT → SDF) if you only have PDBQT, "
        "or copy the missing files into /workspace/d2_system/"
    )
print("\n✓ All input files present")


### Cell 3b (optional) — convert docked PDBQT to SDF

Only run this if you don't already have `levetiracetam_docked.sdf` and need to make
it from the Vina output PDBQT. Skip otherwise.


In [ ]:
import shutil, sys
print(sys.executable)
print(shutil.which("obabel"))

In [ ]:
import os, shutil
os.environ["PATH"] = "/workspace/miniforge3/envs/mddock/bin:" + os.environ["PATH"]
print(shutil.which("obabel"))

In [ ]:
# Cell 3b (optional): Convert docked PDBQT to SDF, keeping the best pose
# Skip if levetiracetam_docked.sdf already exists.
import subprocess

DOCKED_PDBQT = WORKDIR / "levetiracetam_d2_docked.pdbqt"  # ← edit path if different
OUT_SDF      = WORKDIR / "levetiracetam_docked.sdf"

if OUT_SDF.exists():
    print(f"{OUT_SDF.name} already exists — skipping conversion.")
else:
    # Take only the first model (best Vina pose) and convert to SDF
    cmd = ["obabel", str(DOCKED_PDBQT), "-O", str(OUT_SDF),
           "-f", "1", "-l", "1", "--gen3d", "no"]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
        raise RuntimeError("obabel conversion failed — install with: conda install -c conda-forge openbabel")
    print(f"✓ Wrote {OUT_SDF}")


---
## Part 2 — Load the equilibrated D2 + membrane (ParmEd)

We load:
1. `gmx_top.pkl` — the CHARMM36 ParmEd Structure (atoms, bonds, parameters)
2. `day9_final_equilibrated.xml` — positions, velocities, box vectors at end of Day 9

Then we **overwrite** the ParmEd structure's positions and box with the equilibrated values.
This is the cleanest way to carry equilibration forward.


In [ ]:
# Cell 4: Load ParmEd structure of D2 + membrane
t0 = time.time()
with open("gmx_top.pkl", "rb") as f:
    d2_struct = pickle.load(f)
print(f"Loaded gmx_top.pkl in {time.time()-t0:.1f}s")

print(f"  Atoms     : {len(d2_struct.atoms):,}")
print(f"  Residues  : {len(d2_struct.residues):,}")
print(f"  Bonds     : {len(d2_struct.bonds):,}")
print(f"  Box (Å)   : {d2_struct.box}")

# Residue composition sanity check
from collections import Counter
res_counts = Counter(r.name for r in d2_struct.residues)
print("\nTop residue types:")
for name, n in sorted(res_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"    {name:6s} : {n}")

# Expect: TIP3 (~22k), DPPC (~360), CLA (~120), SOD (~110), then ~333 protein residues


In [ ]:
# Cell 5: Pull equilibrated positions and box from the Day 9 State XML
print("Loading day9_final_equilibrated.xml ...")
t0 = time.time()
with open("day9_final_equilibrated.xml") as f:
    equil_state = XmlSerializer.deserialize(f.read())
print(f"  Deserialized in {time.time()-t0:.1f}s")

equil_positions = equil_state.getPositions(asNumpy=True)    # (N, 3) Quantity in nm
equil_box       = equil_state.getPeriodicBoxVectors(asNumpy=True)  # (3, 3) Quantity in nm
equil_vels      = equil_state.getVelocities(asNumpy=True)   # we'll reuse these

print(f"  Positions shape : {equil_positions.shape}")
print(f"  Box (nm)        :")
for v in equil_box:
    print(f"    {v}")

# Sanity check
assert len(equil_positions) == len(d2_struct.atoms), (
    f"Atom count mismatch: ParmEd has {len(d2_struct.atoms)}, "
    f"State has {len(equil_positions)}"
)

# Apply equilibrated positions and box to ParmEd structure (so the merge later uses
# the equilibrated geometry, not the starting MemProtMD geometry)
d2_struct.coordinates = equil_positions.value_in_unit(u.angstrom)  # ParmEd uses Å
d2_struct.box = [
    equil_box[0][0].value_in_unit(u.angstrom),
    equil_box[1][1].value_in_unit(u.angstrom),
    equil_box[2][2].value_in_unit(u.angstrom),
    90.0, 90.0, 90.0,
]
print(f"\n✓ D2 structure updated to equilibrated state")
print(f"  Box (Å) : {d2_struct.box}")


---
## Part 3 — Parameterize levetiracetam with OpenFF

We load the **docked** pose (with its real 3D coordinates from Vina), parameterize with
**OpenFF-2.1.0** (Sage), and convert the resulting Interchange into a ParmEd Structure
so we can merge it with the D2 structure.

The trick: OpenFF Interchange writes GROMACS `.top` + `.gro` files cleanly, and ParmEd
can read those into a Structure that carries all the OpenFF parameters as proper
ForceField terms.


In [ ]:
# Cell 6 (FIXED v3): Build from SMILES, map heavy atoms to docked pose via lookup table
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

LEV_SMILES = "CC[C@@H](N1CCCC1=O)C(N)=O"
DOCKED_PDB = "docking/best_pose_equilibrated_ligand.pdb"

# --- Step 1: Parse the docked PDB manually ---
docked_heavy_coords = []
docked_heavy_elements = []
with open(DOCKED_PDB) as f:
    for line in f:
        if not line.startswith(("ATOM", "HETATM")):
            continue
        atom_name = line[12:16].strip()
        x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        if atom_name.startswith("H"):
            continue
        docked_heavy_coords.append([x, y, z])
        docked_heavy_elements.append(atom_name)
docked_heavy_coords = np.array(docked_heavy_coords)
print(f"Docked heavy atoms : {len(docked_heavy_coords)} {docked_heavy_elements}")
assert len(docked_heavy_coords) == 12

# --- Step 2: Build clean levetiracetam from SMILES ---
mol = Chem.MolFromSmiles(LEV_SMILES)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)
print(f"Clean ligand built : {mol.GetNumAtoms()} atoms ({mol.GetNumHeavyAtoms()} heavy)")

# --- Step 3: Manual PDB→SMILES heavy-atom mapping ---
# Element check (PDB and SMILES element strings should both be 12 atoms)
smiles_heavy_elements = [a.GetSymbol() for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
print(f"SMILES heavy elems : {smiles_heavy_elements}")
print(f"PDB    heavy elems : {docked_heavy_elements}")

# Mapping: PDB heavy index i → SMILES heavy index pdb_to_smiles[i]
# Derived by chemical role identification (chiral C, ethyl tail, amide group, ring atoms)
#   PDB 0  C  chiral C*           → SMILES 2  C*
#   PDB 1  C  ethyl Cβ            → SMILES 1  Cβ
#   PDB 2  C  ethyl Cγ (CH3)      → SMILES 0  Cγ
#   PDB 3  C  amide carbonyl      → SMILES 9  amide C
#   PDB 4  O  amide =O            → SMILES 11 amide =O
#   PDB 5  N  amide NH2           → SMILES 10 amide N
#   PDB 6  N  ring N              → SMILES 3  ring N
#   PDB 7  C  ring Cα' (N–CH2)    → SMILES 4  ring C
#   PDB 8  C  ring C=O            → SMILES 7  ring C=O
#   PDB 9  C  ring Cβ'            → SMILES 5  ring C
#   PDB 10 C  ring Cγ'            → SMILES 6  ring C
#   PDB 11 O  ring =O             → SMILES 8  ring =O
pdb_to_smiles = [2, 1, 0, 9, 11, 10, 3, 4, 7, 5, 6, 8]

# Verify the mapping is element-consistent
for i_pdb, i_smi in enumerate(pdb_to_smiles):
    pdb_el = docked_heavy_elements[i_pdb]
    smi_el = smiles_heavy_elements[i_smi]
    assert pdb_el == smi_el, (
        f"Mapping mismatch at PDB idx {i_pdb}: "
        f"PDB has '{pdb_el}' but SMILES idx {i_smi} has '{smi_el}'"
    )
print("✓ Mapping is element-consistent")

# --- Step 4: Apply docked coordinates to the SMILES molecule's heavy atoms ---
conf = mol.GetConformer()
# In `mol`, heavy atoms are indices 0–11 (H's are 12–25, added afterwards)
for i_pdb, i_smi in enumerate(pdb_to_smiles):
    x, y, z = docked_heavy_coords[i_pdb]
    conf.SetAtomPosition(i_smi, (float(x), float(y), float(z)))

# Constrained-minimize: fix heavy atoms, relax hydrogens around them
mp = AllChem.MMFFGetMoleculeProperties(mol)
ff = AllChem.MMFFGetMoleculeForceField(mol, mp)
heavy_idx_in_mol = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
for i in heavy_idx_in_mol:
    ff.AddFixedPoint(i)
ff.Minimize(maxIts=500)
print("✓ Hydrogens relaxed around fixed docked heavy-atom positions")

# --- Step 5: Sanity-check bond lengths after coordinate transfer ---
for bond in mol.GetBonds():
    a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
    if a1.GetAtomicNum() == 1 or a2.GetAtomicNum() == 1:
        continue  # skip C-H, N-H
    p1 = conf.GetAtomPosition(a1.GetIdx())
    p2 = conf.GetAtomPosition(a2.GetIdx())
    d = ((p1.x-p2.x)**2 + (p1.y-p2.y)**2 + (p1.z-p2.z)**2)**0.5
    if not (0.9 < d < 1.9):
        print(f"  ⚠ Bond {a1.GetSymbol()}{a1.GetIdx()}–{a2.GetSymbol()}{a2.GetIdx()}: {d:.2f} Å (suspicious)")
print("✓ All heavy-atom bond lengths within 0.9–1.9 Å range")

# --- Step 6: Convert to OpenFF and assign charges ---
ligand_off = Molecule.from_rdkit(mol)

from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
nagl = NAGLToolkitWrapper()
nagl.assign_partial_charges(
    molecule=ligand_off,
    partial_charge_method="openff-gnn-am1bcc-0.1.0-rc.3.pt",
)
print(f"Charges assigned via NAGL "
      f"(sum = {sum(c.m for c in ligand_off.partial_charges):.3f} e)")

print(f"\nLigand atoms     : {ligand_off.n_atoms}")
print(f"Ligand bonds     : {ligand_off.n_bonds}")
print(f"SMILES (mapped)  : {ligand_off.to_smiles()}")
print(f"Total formal chg : {sum(a.formal_charge.m for a in ligand_off.atoms)}")
assert ligand_off.n_atoms == 26

docked_coords_A = ligand_off.conformers[0].m_as("angstrom")
print(f"Final ligand centroid (Å): {docked_coords_A.mean(axis=0)}")
print(f"Docked PDB centroid (Å)  : {docked_heavy_coords.mean(axis=0)}")
print("\n✓ Levetiracetam ready at docked pose")

In [ ]:
# Cell 7: OpenFF parameterization → Interchange → GROMACS files → ParmEd
print("Parameterizing with OpenFF Sage 2.1.0 ...")
t0 = time.time()

off_ff = OFFForceField("openff-2.1.0.offxml")
interchange = off_ff.create_interchange(
    topology=ligand_off.to_topology(),
    charge_from_molecules=[ligand_off],
)
print(f"  Interchange built in {time.time()-t0:.1f}s")

# Assign a placeholder box (required for GROMACS export; ParmEd merge overrides it later)
import numpy as np
from openff.units import unit as off_unit
interchange.box = np.eye(3) * 5.0 * off_unit.nanometer

LIG_TOP = WORKDIR / "_tmp_levetiracetam.top"
LIG_GRO = WORKDIR / "_tmp_levetiracetam.gro"
interchange.to_top(str(LIG_TOP))
interchange.to_gro(str(LIG_GRO))
print(f"  Wrote {LIG_TOP.name} and {LIG_GRO.name}")

# (rest of the cell unchanged — load into ParmEd, force coords, rename to LEV)
ligand_struct = pmd.load_file(str(LIG_TOP), xyz=str(LIG_GRO))
print(f"\nLigand ParmEd Structure:")
print(f"  Atoms     : {len(ligand_struct.atoms)}")
print(f"  Bonds     : {len(ligand_struct.bonds)}")
print(f"  Angles    : {len(ligand_struct.angles)}")
print(f"  Dihedrals : {len(ligand_struct.dihedrals)}")
print(f"  Coords (Å) centroid: {ligand_struct.coordinates.mean(axis=0)}")

ligand_struct.coordinates = docked_coords_A
print(f"  Coords after re-setting to docked pose: "
      f"centroid {ligand_struct.coordinates.mean(axis=0)}")

for r in ligand_struct.residues:
    r.name = "LEV"

---
## Part 4 — Merge D2 + levetiracetam at the ParmEd level

ParmEd's `+` operator concatenates two structures, including all their force-field terms.
We must:
1. Concatenate the structures (`d2_struct + ligand_struct`)
2. Preserve the equilibrated box (concatenation can drop it)
3. Sanity-check atom count


In [ ]:
# Cell 8: Combine the two ParmEd structures
combined = d2_struct + ligand_struct

# ParmEd '+' does not always carry over box info — set it explicitly
combined.box = d2_struct.box

n_d2  = len(d2_struct.atoms)
n_lig = len(ligand_struct.atoms)
n_tot = len(combined.atoms)

print(f"D2 atoms       : {n_d2:,}")
print(f"Ligand atoms   : {n_lig}")
print(f"Combined atoms : {n_tot:,}")
print(f"Expected       : {n_d2 + n_lig:,}")
assert n_tot == n_d2 + n_lig, "Atom count mismatch after merge!"

# Residue summary
res_counts = Counter(r.name for r in combined.residues)
print("\nCombined residue summary:")
for name, n in sorted(res_counts.items(), key=lambda x: -x[1])[:12]:
    print(f"    {name:6s} : {n}")
assert res_counts.get("LEV", 0) == 1, "LEV residue not found in combined structure"
print("\n✓ Merge successful, LEV present")
print(f"\nFinal box (Å): {combined.box}")


---
## Part 5 — Build the OpenMM System

This is the single `createSystem()` call that converts the merged ParmEd structure into
a fully-parameterized OpenMM System with:
- All CHARMM36 forces for D2/DPPC/water/ions (HarmonicBond, HarmonicAngle, PeriodicTorsion,
  CustomTorsion for impropers, CMAPTorsion, NonbondedForce + CustomNonbondedForce for NBFix)
- All OpenFF forces for levetiracetam (merged into the same Force objects)

We add a **`MonteCarloMembraneBarostat`** — NOT a plain `MonteCarloBarostat`. Membrane
systems need semi-isotropic pressure coupling: x and y scale together (in the membrane
plane), z scales independently (perpendicular to membrane). Using the wrong barostat
crushes or stretches your bilayer.


In [ ]:
# Cell 9: Build OpenMM System from merged ParmEd structure
print("Creating OpenMM system ...")
t0 = time.time()
system = combined.createSystem(
    nonbondedMethod  = app.PME,
    nonbondedCutoff  = 1.0 * u.nanometer,
    constraints      = app.HBonds,
    rigidWater       = True,
    removeCMMotion   = True,
    ewaldErrorTolerance = 5e-4,
)
print(f"  System built in {time.time()-t0:.1f}s")
print(f"  Particles : {system.getNumParticles():,}")
print(f"  Forces:")
for i in range(system.getNumForces()):
    f = system.getForce(i)
    print(f"    {i}: {f.__class__.__name__}")

# Add membrane barostat — Z is perpendicular to bilayer in MemProtMD setup
# 1 atm in xy plane, 0 surface tension (allowed to expand/contract freely)
barostat = MonteCarloMembraneBarostat(
    1.0 * u.bar,                                       # pressure
    0.0 * u.bar * u.nanometer,                         # surface tension (γ=0 → tensionless)
    310 * u.kelvin,                                    # temperature
    MonteCarloMembraneBarostat.XYIsotropic,            # xy scale together
    MonteCarloMembraneBarostat.ZFree,                  # z scales independently
    25,                                                # frequency (steps)
)
system.addForce(barostat)
print(f"\n✓ Added MonteCarloMembraneBarostat (XYIsotropic, ZFree, 310 K, 1 bar, γ=0)")


In [ ]:
# Cell 10: Build the Simulation object on the CUDA platform
integrator = LangevinMiddleIntegrator(
    310 * u.kelvin,
    1.0 / u.picosecond,
    0.002 * u.picosecond,    # 2 fs timestep (safe with HBonds constraints)
)

platform   = Platform.getPlatformByName("CUDA")
properties = {"Precision": "mixed", "DeviceIndex": "0"}

# Get OpenMM topology from ParmEd (this is what Simulation needs)
omm_topology = combined.topology

simulation = app.Simulation(omm_topology, system, integrator, platform, properties)

# Set initial positions from the merged ParmEd structure (D2 equilibrated + LEV at docked pose)
simulation.context.setPositions(combined.positions)

# Set box vectors
box_nm = np.array(combined.box[:3]) / 10.0  # Å → nm
simulation.context.setPeriodicBoxVectors(
    Vec3(box_nm[0], 0, 0) * u.nanometer,
    Vec3(0, box_nm[1], 0) * u.nanometer,
    Vec3(0, 0, box_nm[2]) * u.nanometer,
)

print("Simulation ready.")
print(f"  Platform : {simulation.context.getPlatform().getName()}")
print(f"  Atoms    : {system.getNumParticles():,}")
print(f"  dt       : 2 fs")
print(f"  T        : 310 K   ;   P : 1 bar (membrane barostat)")


In [ ]:
# ---- Replicate production run --------------------------------------------
REPLICATE_ID = 2          # change to 3 for the next run
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open("day12_equil2.xml") as f:
    st = XmlSerializer.deserialize(f.read())

# transfer only geometry, not the saved force parameters
simulation.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
simulation.context.setPositions(st.getPositions())

# independently seeded velocities — this is what makes the replicate independent
simulation.context.setVelocitiesToTemperature(310 * u.kelvin, 1000 + REPLICATE_ID)
simulation.currentStep = 0
print(f"Replicate {REPLICATE_ID}: velocities re-drawn with seed {1000+REPLICATE_ID}")

simulation.reporters.clear()
simulation.reporters.append(app.DCDReporter(f"day12_production_{tag}.dcd", 25_000))
simulation.reporters.append(app.CheckpointReporter(f"day12_production_{tag}.chk", 500_000))
simulation.reporters.append(app.StateDataReporter(
    f"day12_production_{tag}.log", 5_000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
simulation.saveState(f"day12_production_{tag}_final.xml")

In [ ]:
# ---- Replicate production run --------------------------------------------
REPLICATE_ID = 3          # change to 3 for the next run
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open("day12_equil2.xml") as f:
    st = XmlSerializer.deserialize(f.read())

# transfer only geometry, not the saved force parameters
simulation.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
simulation.context.setPositions(st.getPositions())

# independently seeded velocities — this is what makes the replicate independent
simulation.context.setVelocitiesToTemperature(310 * u.kelvin, 1000 + REPLICATE_ID)
simulation.currentStep = 0
print(f"Replicate {REPLICATE_ID}: velocities re-drawn with seed {1000+REPLICATE_ID}")

simulation.reporters.clear()
simulation.reporters.append(app.DCDReporter(f"day12_production_{tag}.dcd", 25_000))
simulation.reporters.append(app.CheckpointReporter(f"day12_production_{tag}.chk", 500_000))
simulation.reporters.append(app.StateDataReporter(
    f"day12_production_{tag}.log", 5_000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
simulation.saveState(f"day12_production_{tag}_final.xml")

---
## Part 6 — Minimize and equilibrate

The ligand was inserted at a Vina pose with no awareness of the equilibrated D2 side-chain
conformations or water positions — so there will be small clashes. We:

1. **Minimize** — relax clashes (~30 s)
2. **NVT-like warm step** — short, just to get velocities at 310 K
3. **NPT with ligand restrained** (1 ns) — let water/protein adjust around the ligand
4. **NPT unrestrained** (1 ns) — final settle

The ligand restraints are positional (harmonic, k = 1000 kJ/mol·nm²) on each ligand
heavy atom, anchored to its docked coordinate. This prevents the ligand from being
displaced during initial water re-arrangement.


# Cell 12 (FIXED): Add positional restraints on ligand heavy atoms
# ParmEd here exposes `atom.element` as the atomic number (int), not an Element object.

# Find ligand heavy atoms in the combined topology
lig_heavy_idx = []
for atom in combined.atoms:
    if atom.residue.name == "LEV" and atom.atomic_number > 1:
        lig_heavy_idx.append(atom.idx)
print(f"Ligand heavy atoms: {len(lig_heavy_idx)} (expected 12 for levetiracetam)")
assert len(lig_heavy_idx) == 12, "Wrong heavy-atom count for LEV"

# Harmonic positional restraint: 0.5 * k * (periodic distance from anchor)^2
restraint_force = CustomExternalForce("0.5*k*periodicdistance(x,y,z,x0,y0,z0)^2")
restraint_force.addGlobalParameter("k", 1000.0 * u.kilojoule_per_mole/u.nanometer**2)
restraint_force.addPerParticleParameter("x0")
restraint_force.addPerParticleParameter("y0")
restraint_force.addPerParticleParameter("z0")

# Anchor positions = current minimized coordinates
current_positions = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)
for idx in lig_heavy_idx:
    p = current_positions[idx].value_in_unit(u.nanometer)
    restraint_force.addParticle(idx, [p[0], p[1], p[2]])

restraint_force_index = system.addForce(restraint_force)
simulation.context.reinitialize(preserveState=True)
print(f"✓ Added ligand restraints (k=1000 kJ/mol·nm², force index {restraint_force_index})")
print(f"  Restrained atom indices: {lig_heavy_idx}")

In [ ]:
simulation.saveState("day12_preprod_safe.xml")
simulation.saveCheckpoint("day12_preprod_safe.chk")

---
## Part 7 — Production MD (50 ns)

Frames saved every 50 ps → 1000 frames over 50 ns. The DCD is small enough for trajectory
analysis but dense enough to resolve ligand–residue contacts.

If you need to **stop and resume**, kill the cell, then later reload from
`day12_production.chk` (see Cell 17) and call `simulation.step()` for the remaining steps.

**Expected wall time:** ~10–15 hours on RTX PRO 4500 Blackwell for this system size.


In [ ]:
import os, datetime
from pathlib import Path

WORKDIR = Path("/workspace/d2_system")
files = [
    "day12_production.dcd",
    "day12_production.log",
    "day12_production.chk",
    "day12_production_final.xml",
]

print(f"{'File':<35s} {'Size':>10s}   Last modified")
print("─" * 75)
for f in files:
    p = WORKDIR / f
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        mtime = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        print(f"{f:<35s} {size_mb:>8.1f}MB   {mtime:%Y-%m-%d %H:%M:%S}")
    else:
        print(f"{f:<35s} {'MISSING':>10s}")

## Part 10 — Trajectory Analysis (Day 13)

In [ ]:
# What residue name did the PDB end up using for levetiracetam?
# Look at the last few residues (ligand is at the end of the topology)
print("Last 5 residues in topology:")
for res in list(traj.topology.residues)[-5:]:
    n_atoms = res.n_atoms
    print(f"  {res.name:6s} idx={res.index:6d}  ({n_atoms} atoms)")

# Also try alternative names
for name in ["LEV", "UNK", "MOL", "LIG", "UNL"]:
    sel = traj.topology.select(f"resname {name}")
    print(f"  resname {name:5s} → {len(sel)} atoms")

In [ ]:
# Day 13 analyses — run on the 50 ns production trajectory
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as mdt

# Already loaded: traj (1000 frames, 118,052 atoms)
# Ligand residue name in this topology = "UNK"
LIG = "resname UNK"

# Reference frame = frame 0 of production (start of unrestrained dynamics)
ref = traj[0]
print(f"Reference frame: production t=0 (= end of equilibration)")

# Time axis
time_ns = np.arange(traj.n_frames) * 0.05  # 50 ps per frame

# ── Analysis 1: Ligand RMSD vs starting pose ──
protein_ca = traj.topology.select("protein and name CA")
ligand_heavy = traj.topology.select(f"{LIG} and not type H")

# Align entire trajectory on protein Cα, then measure ligand displacement
traj.superpose(ref, atom_indices=protein_ca)
rmsd_lig = mdt.rmsd(traj, ref, atom_indices=ligand_heavy) * 10.0  # nm → Å

# ── Analysis 2: Min ligand–ASP80 distance ──
# ASP80 = residue index 79 (0-indexed); carboxylate oxygens are OD1 and OD2
asp80_o = traj.topology.select("resid 79 and (name OD1 or name OD2)")
lig_polar = traj.topology.select(f"{LIG} and (type O or type N)")
print(f"ASP80 carboxylate O atoms : {len(asp80_o)} (expected 2)")
print(f"Ligand polar atoms (O,N) : {len(lig_polar)} (expected 4: 2N + 2O)")

pairs_asp = np.array([[l, o] for l in lig_polar for o in asp80_o])
d_asp = mdt.compute_distances(traj, pairs_asp) * 10.0  # → Å
min_d_asp = d_asp.min(axis=1)  # per-frame minimum distance to any ASP80 oxygen

# ── Analysis 3: Ligand–protein heavy-atom contact count ──
prot_heavy = traj.topology.select("protein and not type H")
pairs_contact = np.array([[l, p] for l in ligand_heavy for p in prot_heavy])
# This can be slow — use neighbor lists for big systems
contacts = mdt.compute_neighbors(traj, cutoff=0.4,        # 4 Å in nm
                                  query_indices=ligand_heavy,
                                  haystack_indices=prot_heavy)
n_contacts = np.array([len(c) for c in contacts])

# ── Analysis 4: ASP80 H-bond persistence ──
# H-bond criterion: donor heavy atom within 3.5 Å of ASP80 O, AND donor-H...O angle > 120°
# Levetiracetam donors = the 2 N-H of the primary amide (N15 in UNK)
lig_donors = traj.topology.select(f"{LIG} and type N")
# Find H atoms bonded to ligand N (we need both N and its H for angle calculation)
print(f"Ligand N atoms : {len(lig_donors)}")

# Use simple distance + angle criterion via Baker-Hubbard or manual computation
# Quick version: just track frames where ANY ligand polar atom is < 3.5 Å of ASP80 O
asp80_hbond_frames = (min_d_asp < 3.5).astype(int)
hbond_pct = 100.0 * asp80_hbond_frames.mean()

# ── Make the master 2x2 figure ──
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) Ligand RMSD
ax = axes[0, 0]
ax.plot(time_ns, rmsd_lig, lw=1.0, color="#1f77b4")
ax.axhline(3.0, ls="--", c="gray", lw=0.8, label="3 Å threshold")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Ligand RMSD (Å)")
ax.set_title("(a) Levetiracetam RMSD vs starting pose")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)

# (b) ASP80 distance
ax = axes[0, 1]
ax.plot(time_ns, min_d_asp, lw=1.0, color="#d62728")
ax.axhline(3.5, ls="--", c="gray", lw=0.8, label="H-bond cutoff (3.5 Å)")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Min distance (Å)")
ax.set_title("(b) Lev polar atom → ASP80 carboxylate")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)

# (c) Contact count
ax = axes[1, 0]
ax.plot(time_ns, n_contacts, lw=1.0, color="#2ca02c")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("# protein heavy atoms within 4 Å")
ax.set_title("(c) Ligand–protein contact count")
ax.grid(alpha=0.3)

# (d) Distribution histogram of ASP80 distance
ax = axes[1, 1]
ax.hist(min_d_asp, bins=40, color="#9467bd", edgecolor="black", alpha=0.75)
ax.axvline(3.5, ls="--", c="gray", lw=0.8, label="3.5 Å")
ax.set_xlabel("Min lev–ASP80 distance (Å)")
ax.set_ylabel("Frames")
ax.set_title("(d) ASP80 distance distribution")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("day12_analysis_master.png", dpi=150)
plt.show()

# ── Print the key numbers ──
print("\n" + "═" * 60)
print("DAY 12 — D2 + LEVETIRACETAM 50 ns MD — KEY RESULTS")
print("═" * 60)
print(f"\nLigand RMSD (Å):")
print(f"  Mean       : {rmsd_lig.mean():.2f}")
print(f"  Max        : {rmsd_lig.max():.2f}")
print(f"  Final (50ns): {rmsd_lig[-1]:.2f}")
print(f"  % time <3Å : {100*(rmsd_lig < 3).mean():.1f}%")
print(f"  % time <5Å : {100*(rmsd_lig < 5).mean():.1f}%")

print(f"\nLigand–ASP80 distance (Å):")
print(f"  Mean       : {min_d_asp.mean():.2f}")
print(f"  Min        : {min_d_asp.min():.2f}")
print(f"  Max        : {min_d_asp.max():.2f}")
print(f"  % time <3.5Å (H-bond): {hbond_pct:.1f}%")
print(f"  % time <5.0Å (close) : {100*(min_d_asp < 5.0).mean():.1f}%")

print(f"\nLigand–protein contacts (4 Å cutoff):")
print(f"  Mean # contacts : {n_contacts.mean():.1f}")
print(f"  Min            : {n_contacts.min()}")
print(f"  Max            : {n_contacts.max()}")
print(f"  Frames with 0 contacts (ligand escaped): {(n_contacts == 0).sum()}")
print("═" * 60)

In [ ]:
# Identify the residues making contact with levetiracetam (top 15)
from collections import Counter

# Get per-frame contacts at 4 Å, group by protein residue
print("Computing per-residue contact frequencies across 50 ns ...")
prot_residues = [r for r in traj.topology.residues if r.is_protein]
res_contact_counts = Counter()

# Atom index → residue mapping (for protein only)
atom_to_res = {}
for r in prot_residues:
    for a in r.atoms:
        if a.element.symbol != "H":
            atom_to_res[a.index] = r

# Use the contact lists we already computed (in `contacts` from previous cell)
# `contacts[i]` is the list of haystack atom indices within 4 Å of ANY ligand heavy atom at frame i
for frame_contacts in contacts:
    residues_in_frame = set()
    for atom_idx in frame_contacts:
        if atom_idx in atom_to_res:
            residues_in_frame.add(atom_to_res[atom_idx])
    for r in residues_in_frame:
        res_contact_counts[r] += 1

# Top 15 residues by contact frequency
print(f"\n{'Rank':<6}{'Residue':<12}{'Contact %':<12}{'PDB resid':<12}")
print("─" * 50)
top15 = res_contact_counts.most_common(15)
for rank, (res, count) in enumerate(top15, 1):
    pct = 100 * count / traj.n_frames
    # PDB residue ID — MDTraj uses 0-indexed internal IDs but PDB numbering is separate
    print(f"{rank:<6}{res.name + str(res.resSeq):<12}{pct:>6.1f}%      {res.resSeq:<12}")

# Save to CSV for later
import csv
with open("day12_residue_contacts.csv", "w") as f:
    w = csv.writer(f)
    w.writerow(["rank", "residue_name", "residue_id_pdb", "contact_percent"])
    for rank, (res, count) in enumerate(res_contact_counts.most_common(50), 1):
        w.writerow([rank, res.name, res.resSeq, 100*count/traj.n_frames])
print("\n✓ Saved top 50 residues to day12_residue_contacts.csv")

In [ ]:
# Which residue's polar atoms are H-bond-distance from the ligand?
print("Computing per-residue H-bond candidate frequencies (< 3.5 Å, polar atoms only) ...")

# Ligand polar atoms (already have these from previous cell)
lig_polar = traj.topology.select(f"{LIG} and (type O or type N)")

# All protein polar atoms (O, N) that could H-bond
prot_polar = traj.topology.select("protein and (type O or type N) and not type H")
print(f"Ligand polar atoms : {len(lig_polar)}")
print(f"Protein polar atoms: {len(prot_polar)}")

# Compute all ligand-polar × protein-polar distances
pairs = np.array([[l, p] for l in lig_polar for p in prot_polar])
print(f"Computing {len(pairs):,} pair distances across {traj.n_frames} frames...")
all_d = mdt.compute_distances(traj, pairs) * 10.0  # → Å

# For each frame, find which protein atoms are < 3.5 Å of any ligand polar
hbond_frames_per_res = Counter()
prot_polar_to_res = {}
for atom_idx in prot_polar:
    a = traj.topology.atom(atom_idx)
    prot_polar_to_res[atom_idx] = a.residue

# For each frame, count residues with at least one polar atom within 3.5 Å
for frame_idx in range(traj.n_frames):
    frame_d = all_d[frame_idx].reshape(len(lig_polar), len(prot_polar))
    # For each protein polar atom, is it within 3.5 Å of any ligand polar?
    close = (frame_d < 3.5).any(axis=0)  # shape: (n_prot_polar,)
    residues_close = set()
    for atom_pos, atom_idx in enumerate(prot_polar):
        if close[atom_pos]:
            residues_close.add(prot_polar_to_res[atom_idx])
    for r in residues_close:
        hbond_frames_per_res[r] += 1

# Top 10 H-bond candidates
print(f"\n{'Rank':<6}{'Residue':<14}{'H-bond %':<12}{'Class':<20}")
print("─" * 60)
chem_class = {
    "SER":"H-bond donor/acceptor (OH)", "THR":"H-bond donor/acceptor (OH)",
    "TYR":"H-bond + π", "HIS":"H-bond + π", "ASP":"H-bond acceptor (COO⁻)",
    "GLU":"H-bond acceptor (COO⁻)", "ASN":"H-bond donor/acceptor (CONH₂)",
    "GLN":"H-bond donor/acceptor (CONH₂)", "TRP":"H-bond + π (indole NH)",
    "ARG":"H-bond donor (guanidinium)", "LYS":"H-bond donor (NH₃⁺)",
    "CYS":"weak H-bond (SH)"
}
for rank, (res, count) in enumerate(hbond_frames_per_res.most_common(10), 1):
    pct = 100 * count / traj.n_frames
    cls = chem_class.get(res.name, "backbone or other")
    print(f"{rank:<6}{res.name + str(res.resSeq):<14}{pct:>6.1f}%      {cls}")

In [ ]:
# Pull residue NAMES at canonical orthosteric positions, assuming offset of +34
# If your sys uses an offset, residue at position 80 - 34 = 46 should be ASP if numbering is canonical
# Instead just print the FULL sequence and find ASP residues
asp_residues = [r for r in traj.topology.residues if r.is_protein and r.name == "ASP"]
print(f"All ASP residues in topology: {[r.resSeq for r in asp_residues]}")
trp_residues = [r for r in traj.topology.residues if r.is_protein and r.name == "TRP"]
print(f"All TRP residues in topology: {[r.resSeq for r in trp_residues]}")
his_residues = [r for r in traj.topology.residues if r.is_protein and r.name == "HIS"]
print(f"All HIS residues in topology: {[r.resSeq for r in his_residues]}")

In [ ]:
# Proper H-bond test: heavy atom < 3.5 Å AND donor-H...acceptor angle > 120°
hbonds_baker_hubbard = mdt.baker_hubbard(traj, freq=0.1, periodic=True, distance_cutoff=0.35, angle_cutoff=2.094)  # 120° in radians

# Filter for ligand involvement
lig_atoms = set(traj.topology.select(f"{LIG}"))
lev_hbonds = [(d, h, a) for d, h, a in hbonds_baker_hubbard 
              if (d in lig_atoms) or (a in lig_atoms)]

print(f"H-bonds with proper geometry (≥10% persistence): {len(lev_hbonds)}")
for d, h, a in lev_hbonds:
    da, ha, aa = traj.topology.atom(d), traj.topology.atom(h), traj.topology.atom(a)
    print(f"  {da.residue.name}{da.residue.resSeq}:{da.name} — H — "
          f"{aa.residue.name}{aa.residue.resSeq}:{aa.name}")

In [ ]:
# Save a representative middle frame as PDB for PyMOL/ChimeraX rendering
mid_frame = traj[500]  # frame at 25 ns
mid_frame.save_pdb("day12_representative_frame_25ns.pdb")
print("✓ Saved representative frame for visualization")

In [ ]:
import os, numpy as np
import mdtraj as mdt
import matplotlib.pyplot as plt

os.chdir("/workspace/d2_system")

# Use the equilibrated PDB topology — we need one of these for MDTraj to read the DCD
# The day12_minimized.pdb has the LEV residue and matches the trajectory's atom count
top_file = "day12_minimized.pdb"
print(f"Loading trajectory ({1416.7:.0f} MB, 1000 frames) ...")
traj = mdt.load("day12_production.dcd", top=top_file)
print(f"  ✓ Loaded: {traj.n_frames} frames, {traj.n_atoms:,} atoms")
print(f"  Time span: {traj.n_frames * 0.05:.1f} ns (50 ps per frame)")

# Quick selections check
n_lev = len(traj.topology.select("resname LEV"))
n_ca = len(traj.topology.select("protein and name CA"))
print(f"  LEV atoms found: {n_lev} (expected 26)")
print(f"  Protein Cα found: {n_ca} (expected ~333 for D2)")

---
## Part 8 — Resume from checkpoint (reference)

Only use this if production stopped early. It rebuilds the simulation object and
loads the checkpoint to continue. **Do not run during a successful run.**


In [ ]:
# Cell 16: (Reference only) Resume production from checkpoint
# Uncomment and run if you killed Cell 15 before it finished.
#
# integrator = LangevinMiddleIntegrator(310*u.kelvin, 1.0/u.picosecond, 0.002*u.picosecond)
# platform   = Platform.getPlatformByName("CUDA")
# simulation = app.Simulation(combined.topology, system, integrator, platform, {"Precision":"mixed"})
# simulation.loadCheckpoint("day12_production.chk")
# print("Resumed at step", simulation.currentStep)
# # Calculate remaining steps and continue
# remaining = PROD_STEPS - simulation.currentStep
# print(f"Remaining: {remaining:,} steps")
# # Re-attach reporters (they get lost on reload), then simulation.step(remaining)


---
## Part 9 — Quick stability check: ligand RMSD vs starting pose

This is a **first-look** analysis. The full Day 13 analysis (per-residue contacts,
ASP80 H-bond persistence, water bridging, pocket waters, RMSF) belongs in a separate
notebook so you can iterate on plots without re-loading the trajectory.

**What to look for:**
- Ligand RMSD < 3 Å throughout → bound state is stable, ASP80 hypothesis likely holds
- Ligand RMSD rising to 5–10 Å → ligand drifted but stayed in pocket
- Ligand RMSD > 15 Å → ligand exited the pocket (interesting result — D2 doesn't hold lev)


In [ ]:
# Cell 17: Quick ligand RMSD using MDTraj
import mdtraj as mdt
import matplotlib.pyplot as plt

# Save a reference PDB at production t=0 (we'll align to this)
# It's the structure right after equilibration ended.
with open("day12_reference.pdb", "w") as f:
    app.PDBFile.writeFile(
        combined.topology,
        simulation.context.getState(getPositions=True).getPositions(),
        f,
    )

ref = mdt.load("day12_reference.pdb")
traj = mdt.load("day12_production.dcd", top="day12_reference.pdb")
print(f"Trajectory : {traj.n_frames} frames, dt = 50 ps → "
      f"{traj.n_frames*0.05:.1f} ns total")

# Atom selections
protein_ca = traj.topology.select("protein and name CA")
ligand_all = traj.topology.select("resname LEV")

# Align on protein Cα, then measure ligand displacement
traj.superpose(ref, atom_indices=protein_ca)
rmsd_lig = mdt.rmsd(traj, ref, atom_indices=ligand_all) * 10.0  # nm → Å

time_ns = np.arange(traj.n_frames) * 0.05

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_ns, rmsd_lig, lw=1.2)
ax.axhline(3.0, ls="--", c="gray", lw=0.8, label="3 Å threshold")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Ligand RMSD vs t=0 (Å)")
ax.set_title("Levetiracetam stability in D2 binding pocket")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("day12_ligand_rmsd.png", dpi=150)
plt.show()

print(f"\nLigand RMSD summary:")
print(f"  Mean    : {rmsd_lig.mean():.2f} Å")
print(f"  Max     : {rmsd_lig.max():.2f} Å")
print(f"  Final   : {rmsd_lig[-1]:.2f} Å")
print(f"  % time < 3 Å : {100*(rmsd_lig < 3).mean():.1f}%")


---
## What you have at the end of Day 12

| File | What it is |
|---|---|
| `day12_minimized.xml` | Post-minimization state (clashes resolved) |
| `day12_equil1.xml` | After 1 ns restrained NPT |
| `day12_equil2.xml` | After 1 ns unrestrained NPT (ready for production) |
| `day12_production.dcd` | 50 ns production trajectory (1000 frames @ 50 ps) |
| `day12_production.log` | Energy/T/P log every 10 ps |
| `day12_production.chk` | Binary checkpoint (every 1 ns) — for restart |
| `day12_production_final.xml` | Final state — feed into Day 13 analysis or extend |
| `day12_reference.pdb` | Topology + start-of-production positions (for MDTraj loading) |
| `day12_ligand_rmsd.png` | First-look stability plot |

**Day 13 plan (next notebook):**
1. ASP80 H-bond persistence (distance + angle criteria over trajectory)
2. Levetiracetam–residue contact map (residence times per contact)
3. Binding pocket water occupancy
4. Per-residue RMSF on D2 transmembrane helices
5. Comparison with apo D2 (Day 9–10) — does ligand binding alter pocket dynamics?
